## Google Colab and data paths

Set `KDD_PROJECT_ROOT` if your project is not stored at `/content/drive/MyDrive/kdd_dropout_project`. The configured folder must contain `dataset/`, `shared/metadata/`, and `artifacts/`.


In [ ]:
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    pass

def discover_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").exists() and (candidate / "shared" / "metadata").exists():
            return candidate
    return start

default_root = (
    Path("/content/drive/MyDrive/kdd_dropout_project")
    if IN_COLAB else discover_project_root(Path.cwd())
)
PROJECT_ROOT = Path(os.environ.get("KDD_PROJECT_ROOT", default_root))
DATA_ROOT = PROJECT_ROOT / "dataset"
METADATA_ROOT = PROJECT_ROOT / "shared" / "metadata"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ARTIFACT_ROOT = ARTIFACT_ROOT / "fslsm_bilstm"
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifacts: {MODEL_ARTIFACT_ROOT}")


# Model Training + Evaluation: Baseline vs. FSLSM-Fused BiLSTM

Extends the original `03_Model_Training_Evaluation.ipynb`. Same dataset, same model
backbone, same training loop -- the only change is an optional fusion of the 4
window-level FSLSM scores (from `02_Data_Preprocessing_FSLSM.ipynb`) into the
attention-pooled context vector right before the final classifier layer.

Trains and evaluates **both** configurations (`use_fslsm=False` and `use_fslsm=True`)
with identical seed/splits/hyperparameters, then prints a side-by-side comparison
table -- this gives you row 1 (baseline) and row 8 (+ FSLSM scores) of the
consolidated results table directly.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ==========================================
# 1. LOAD PROCESSED DATA (includes FSLSM scores)
# ==========================================
print("Loading processed data...")
data = torch.load(MODEL_ARTIFACT_ROOT / "processed_data_fslsm.pt", weights_only=False)  # Trust only the artifact created by notebook 01

src_train, evt_train, cat_train, gap_train, y_train, fslsm_train = data["train"]
src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test = data["test"]
source_vocab, event_vocab, category_vocab = data["vocabs"]
truth_test = data["truth_test"]

print(f"Train windows: {src_train.shape[0]}, FSLSM feature dim: {fslsm_train.shape[1]}")


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, src, evt, cat, gaps, y, fslsm):
        self.src, self.evt, self.cat = torch.LongTensor(src), torch.LongTensor(evt), torch.LongTensor(cat)
        self.gaps, self.y = torch.FloatTensor(gaps), torch.FloatTensor(y)
        self.fslsm = torch.FloatTensor(fslsm)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.evt[idx], self.cat[idx], self.gaps[idx], self.y[idx], self.fslsm[idx]

def make_loaders(batch_size=128):
    train_ds = SeqDataset(src_train, evt_train, cat_train, gap_train, y_train, fslsm_train)
    test_ds = SeqDataset(src_test, evt_test, cat_test, gap_test, y_test, fslsm_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               generator=torch.Generator().manual_seed(SEED))
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, test_loader


In [ ]:
# ==========================================
# 2. DEFINE MODEL (toggle: use_fslsm)
# ==========================================
class BiLSTMModel(nn.Module):
    def __init__(self, use_fslsm=False, fslsm_dim=4):
        super().__init__()
        self.use_fslsm = use_fslsm
        self.source_emb = nn.Embedding(len(source_vocab) + 1, 4, padding_idx=0)
        self.event_emb = nn.Embedding(len(event_vocab) + 1, 8, padding_idx=0)
        self.category_emb = nn.Embedding(len(category_vocab) + 1, 8, padding_idx=0)

        self.lstm = nn.LSTM(input_size=21, hidden_size=64, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(128, 1)

        fc_in = 128 + (fslsm_dim if use_fslsm else 0)
        self.fc = nn.Linear(fc_in, 1)

    def forward(self, src, evt, cat, gaps, lengths, fslsm=None):
        x = torch.cat([self.source_emb(src), self.event_emb(evt), self.category_emb(cat), gaps.unsqueeze(-1)], dim=2)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = torch.nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True)

        attn_scores = self.attention(out).squeeze(-1)
        attn_scores[~(src[:, :out.size(1)] != 0)] = -1e9  # Mask padding
        context = torch.sum(out * torch.softmax(attn_scores, dim=1).unsqueeze(-1), dim=1)

        if self.use_fslsm:
            context = torch.cat([context, fslsm], dim=1)

        return torch.sigmoid(self.fc(context)).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ==========================================
# 3. TRAIN + EVALUATE (one function, called for each config)
# ==========================================
def train_and_evaluate(use_fslsm, epochs=3, lr=0.001, batch_size=128):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # re-seed for fair comparison

    train_loader, test_loader = make_loaders(batch_size=batch_size)
    model = BiLSTMModel(use_fslsm=use_fslsm).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    tag = "WITH FSLSM" if use_fslsm else "BASELINE (no FSLSM)"
    print(f"\n=== Training {tag} ===")

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for src, evt, cat, gaps, labels, fslsm in train_loader:
            src, evt, cat, gaps, labels, fslsm = [t.to(device) for t in (src, evt, cat, gaps, labels, fslsm)]

            optimizer.zero_grad()
            outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1), fslsm if use_fslsm else None)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

    model.eval()
    window_preds = []
    with torch.no_grad():
        for src, evt, cat, gaps, _, fslsm in test_loader:
            src, evt, cat, gaps, fslsm = [t.to(device) for t in (src, evt, cat, gaps, fslsm)]
            outputs = model(src, evt, cat, gaps, (src != 0).sum(dim=1), fslsm if use_fslsm else None)
            window_preds.extend(outputs.cpu().numpy())

    pred_df = pd.DataFrame({"student_id": test_ids, "pred": window_preds})
    eval_df = pred_df.groupby("student_id")["pred"].mean().reset_index().merge(truth_test, on="student_id")
    eval_df["predicted_dropout"] = (eval_df["pred"] > 0.5).astype(int)

    metrics = {
        "setting": tag,
        "accuracy": accuracy_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "precision": precision_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "recall": recall_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "f1": f1_score(eval_df["dropout"], eval_df["predicted_dropout"]),
        "auc": roc_auc_score(eval_df["dropout"], eval_df["pred"]),
    }
    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1 Score  : {metrics['f1']:.4f}")
    print(f"  AUC       : {metrics['auc']:.4f}")
    return metrics


In [ ]:
# ==========================================
# 4. RUN BOTH CONFIGS
# ==========================================
baseline_metrics = train_and_evaluate(use_fslsm=False)
fslsm_metrics = train_and_evaluate(use_fslsm=True)


In [ ]:
# ==========================================
# 5. COMPARISON TABLE
# ==========================================
comparison = pd.DataFrame([baseline_metrics, fslsm_metrics]).set_index("setting")
comparison = comparison.round(4)
comparison.columns = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
print("\nComparison: Sequence-only vs Sequence + FSLSM scores")
comparison


## Notes for the consolidated results table

- Both rows above use the **identical** window/split/seed -- the only difference is
  whether the 4 FSLSM scores are concatenated into the classifier input. This isolates
  the effect of adding the scores, matching the ask in the project guide.
- To extend this into an ablation (which FSLSM dimension matters most), slice
  `fslsm_train[:, [i]]` / `fslsm_test[:, [i]]` for a single dimension and pass
  `fslsm_dim=1` to `BiLSTMModel`, repeating `train_and_evaluate` per dimension.
- These two rows correspond to rows 1 and 8 of the master comparison table discussed
  earlier (event-level baseline vs. event-level + rule-based FSLSM scores). Rows 6-7
  (cluster-derived learning-style labels) come from the separate `EXP1.pdf` pipeline
  and can be added to this same `comparison` DataFrame once available.